# Process Capability Analysis

This notebook studies the synthetic **BOOTH_01 / A-L1 / Coating Thickness** stream (70 / 75 / 80 µm). It demonstrates method, not certification of a real process.

## 1. Repository Inspection and Baseline Snapshot
The capability work adds a separate input view and Python calculation. The existing I-MR chart formulas and signal rules are checked against `HEAD`; no event or robot table is used for capability.

## 2. Scoped Data Extraction from `v_measurement_enriched`
The SQL input query selects one row per measurement from the enriched view and applies the fixed characteristic, unit, and specification. It does not infer limits from observed values. Run with `psql` and the project's usual `PG*` connection environment or service configuration.

## 3. SPC Signal/Status Reuse and Stability Join
The input view reuses the existing SPC result by `measurement_id` for actual `shift_instance` and `spc_signal`. A signal is an investigation prompt, not proof of a special cause. The complete history is retained.

## 4. Deterministic Ordering and Shift-Bounded Moving Ranges
Sort by `measurement_timestamp, measurement_id`. Within each actual shift instance, the first value has no moving range; later values use the absolute difference from the preceding value. No pair crosses a shift boundary.

## 5. Reusable Capability Function (`Cp`, `Cpk`, `Pp`, `Ppk`)
`src.capability.calculate_capability` is canonical. Within sigma is `MRbar / 1.128`; overall sigma is sample standard deviation with `ddof=1`.

## 6. Prerequisite/Availability Status Handling
Missing or inconsistent limits, insufficient observations, no valid moving-range pair, zero sigma, and near-zero sigma are explicit statuses. Unavailable indices are `None`, never infinity or substituted values. The near-zero guard is 0.001 µm, matching the stored 0.001 µm precision.

## 7. Initial Full-History Capability Result Assembly
The result includes stream/specification identity, descriptive statistics, sigma diagnostics, indices, distances to both limits, strict OOS counts, and SPC signal/interpretation status.

## 8. OOS Semantics and Compliance Metrics
A value is OOS only if strictly below LSL or strictly above USL. Equality to either limit is in specification. OOS rate uses all stream measurements as its denominator.

## 9. Time-Based Descriptive Diagnostics (Month / Shift Instance / Shift Type)
Month, actual shift instance, and shift type summaries are descriptive only. They omit capability indices and impose no arbitrary minimum subgroup size.

## 10. Distribution Diagnostics (Histogram, Q-Q, Mean vs Median, Skewness)
Classical indices assume a normal model; this is an assumption to inspect, not a conclusion. Use the histogram, Q-Q plot, mean/median comparison, and skewness together. A large-sample normality-test p-value is not a sole decision rule.

## 11. SPC Stability vs Capability Interpretation (Phase I Exploratory Labeling)
The stream contains SPC signals. Report full-history indices as **exploratory Phase I**, not proof of capability. Stability and specification compliance are distinct: a stable process can be incapable, and favorable indices do not remove the need to investigate when stability assumptions are violated.

## 12. Manual Fixture Reproduction and Hand-Check Cells
For `[3, 5, 7, 9]`, LSL 0, target 5, USL 10: ranges are `[2, 2, 2]`, `MRbar=2`, within sigma is `2/1.128`, and sample standard deviation is `sqrt(20/3)`. Verify all four indices below.

## 13. Deterministic Regression Tests (A–G)
`tests/test_capability.py` covers the manual fixture, centering, strict boundaries, sigma availability, shift boundaries, deterministic ordering, and descriptive groups. Live reconciliation is in `sql/analysis/validate_capability_inputs.sql`.

## 14. SQL↔Python Reconciliation Checks
Reconcile row count, OOS count/rate, and SPC signal count. The view must pass reconciliation before interpreting the result.

## 15. Documentation Cells for Formulas and Interpretation Rules
Cp/Pp compare spread with specification width; Cpk/Ppk also use distance from the mean to the nearer limit. Cp can exceed Cpk when the mean is off-center. Pp can differ from Cp when overall spread differs from short-term within spread. Control limits describe process behavior; specification limits define acceptance. This synthetic data demonstrates analysis, not real coating-line performance.

## 16. Execution, Validation Outputs, and Change Summary (`git diff --check`)
The final cell prints regression-test, live reconciliation, capability, stability, changed-file, and whitespace-check results. No commit or push is performed.

In [ ]:
from io import StringIO
from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.capability import calculate_capability, describe_capability_groups

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "sql" / "views.sql").exists():
    REPO_ROOT = REPO_ROOT.parent

spc_detail = subprocess.run(
    ["git", "diff", "--quiet", "HEAD", "--", "sql/analysis/spc_rules.sql"],
    cwd=REPO_ROOT,
    check=False,
)
spc_summary = subprocess.run(
    ["git", "diff", "--quiet", "HEAD", "--", "sql/analysis/spc_signal_summary.sql"],
    cwd=REPO_ROOT,
    check=False,
)
head_views = subprocess.run(
    ["git", "show", "HEAD:sql/views.sql"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout
current_views = (REPO_ROOT / "sql" / "views.sql").read_text(encoding="utf-8")
head_spc_view = head_views.split("CREATE OR REPLACE VIEW v_spc_rule_results_booth01_a_l1 AS", 1)[1].split(
    "CREATE OR REPLACE VIEW v_production_volume AS", 1
)[0]
current_spc_view = current_views.split("CREATE OR REPLACE VIEW v_spc_rule_results_booth01_a_l1 AS", 1)[1].split(
    "CREATE OR REPLACE VIEW v_capability_measurements_booth01_a_l1 AS", 1
)[0]
assert spc_detail.returncode == 0, "Existing SPC detail SQL was modified."
assert spc_summary.returncode == 0, "Existing SPC summary SQL was modified."
assert current_spc_view == head_spc_view, "Existing SPC view definition changed."
print("SPC rules, signal summary, and existing SPC view match HEAD.")


database_name = os.environ.get("PGDATABASE", "spc_quality_monitoring")
query_path = REPO_ROOT / "sql" / "analysis" / "capability_inputs.sql"
query_run = subprocess.run(
    ["psql", "-X", "--csv", "-v", "ON_ERROR_STOP=1", "-d", database_name, "-f", str(query_path)],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
measurements = pd.read_csv(StringIO(query_run.stdout))
measurements["measurement_timestamp"] = pd.to_datetime(measurements["measurement_timestamp"])
measurements["production_timestamp"] = pd.to_datetime(measurements["production_timestamp"])
measurements["production_month"] = pd.to_datetime(measurements["production_month"])
measurements["shift_instance"] = pd.to_datetime(measurements["shift_instance"])
assert measurements["measurement_id"].is_unique
assert set(measurements["booth"]) == {"BOOTH_01"}
assert set(measurements["measurement_point"]) == {"A-L1"}
assert set(measurements["lsl"]) == {70.0}
assert set(measurements["target"]) == {75.0}
assert set(measurements["usl"]) == {80.0}
print(f"Loaded {len(measurements):,} unique SQL-derived measurements from {database_name}.")

In [ ]:
ordered = measurements.sort_values(
    ["measurement_timestamp", "measurement_id"],
    kind="mergesort",
).copy()
ordered["moving_range"] = (
    ordered.groupby("shift_instance", sort=False)["measured_value"].diff().abs()
)

capability_result = calculate_capability(measurements)
capability_result

In [ ]:
monthly_diagnostics = describe_capability_groups(measurements, ["production_month"])
shift_instance_diagnostics = describe_capability_groups(
    measurements,
    ["shift_instance", "shift_id", "shift_name"],
)
shift_type_diagnostics = describe_capability_groups(measurements, ["shift_id", "shift_name"])

print("Production month")
display(monthly_diagnostics)
print("Actual shift instance; descriptive only")
display(shift_instance_diagnostics.head())
print("Shift type pooled over instances; descriptive only")
display(shift_type_diagnostics)

In [ ]:
from statistics import NormalDist

values = measurements["measured_value"].astype(float)
mean_value = float(values.mean())
median_value = float(values.median())
overall_sigma_value = float(values.std(ddof=1))
print(
    {
        "mean": mean_value,
        "median": median_value,
        "mean_minus_median": mean_value - median_value,
        "skewness": float(values.skew()),
    }
)

figure, (histogram_axis, qq_axis) = plt.subplots(1, 2, figsize=(13, 5))
histogram_axis.hist(values, bins=35, color="#3b7a78", edgecolor="white")
histogram_axis.axvline(mean_value, color="#cf5935", linestyle="--", label="Mean")
histogram_axis.axvline(median_value, color="#283f61", linestyle=":", label="Median")
histogram_axis.set(title="Coating thickness distribution", xlabel="Thickness (µm)", ylabel="Measurements")
histogram_axis.legend()

standard_normal = NormalDist()
probabilities = (np.arange(len(values)) + 0.5) / len(values)
theoretical_quantiles = np.array([standard_normal.inv_cdf(float(probability)) for probability in probabilities])
sample_quantiles = np.sort(values.to_numpy())
qq_axis.scatter(theoretical_quantiles, sample_quantiles, s=8, alpha=0.45, color="#3b7a78")
qq_axis.plot(
    [theoretical_quantiles.min(), theoretical_quantiles.max()],
    [mean_value + theoretical_quantiles.min() * overall_sigma_value,
     mean_value + theoretical_quantiles.max() * overall_sigma_value],
    color="#cf5935",
    linestyle="--",
)
qq_axis.set(title="Normal Q-Q diagnostic", xlabel="Theoretical normal quantiles", ylabel="Observed quantiles (µm)")
figure.tight_layout()
plt.show()

In [ ]:
manual_fixture = pd.DataFrame(
    {
        "measurement_id": [1, 2, 3, 4],
        "measurement_timestamp": pd.date_range("2026-01-01", periods=4, freq="min"),
        "shift_instance": ["fixture-shift"] * 4,
        "measured_value": [3.0, 5.0, 7.0, 9.0],
        "lsl": [0.0] * 4,
        "target": [5.0] * 4,
        "usl": [10.0] * 4,
        "spc_signal": [False] * 4,
    }
)
manual_result = calculate_capability(manual_fixture)
manual_expected = {
    "mrbar": 2.0,
    "within_sigma": 2 / 1.128,
    "overall_sigma": np.sqrt(20 / 3),
    "Cp": 10 / (6 * (2 / 1.128)),
    "Cpk": 4 / (3 * (2 / 1.128)),
    "Pp": 10 / (6 * np.sqrt(20 / 3)),
    "Ppk": 4 / (3 * np.sqrt(20 / 3)),
}
for metric, expected in manual_expected.items():
    assert np.isclose(manual_result[metric], expected)
print("Hand-checked fixture result:")
{metric: round(manual_result[metric], 3) for metric in manual_expected}

In [ ]:
reconciliation_run = subprocess.run(
    [
        "psql", "-X", "--csv", "-v", "ON_ERROR_STOP=1", "-d", database_name,
        "-f", str(REPO_ROOT / "sql" / "analysis" / "validate_capability_inputs.sql"),
    ],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
sql_reconciliation = pd.read_csv(StringIO(reconciliation_run.stdout)).iloc[0]
assert sql_reconciliation["reconciliation_status"] == "PASS"
assert len(measurements) == capability_result["measurement_count"]
assert int(measurements["is_oos"].sum()) == capability_result["OOS_count"]
assert np.isclose(measurements["is_oos"].mean(), capability_result["OOS_rate"])
assert int(measurements["spc_signal"].sum()) == capability_result["spc_signal_count"]

capability_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_capability.py", "-q"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
diff_check = subprocess.run(
    ["git", "diff", "--check"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
changed_files = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout

report_keys = [
    "measurement_count", "mean", "median", "sample_std", "min", "max",
    "mrbar", "valid_mr_pairs", "d2", "within_sigma", "overall_sigma",
    "below_LSL_count", "above_USL_count", "OOS_count", "OOS_rate",
    "Cp", "Cpk", "Pp", "Ppk", "distance_mean_to_LSL",
    "distance_mean_to_USL", "spc_signal_count", "spc_signal_rate",
    "stability_signal_status", "interpretation_status", "availability_status",
]
print("Capability result:")
print({key: capability_result[key] for key in report_keys})
print("SQL reconciliation:")
print(sql_reconciliation.to_dict())
print("Capability tests:")
print(capability_tests.stdout.strip())
print(f"git diff --check: {'PASS' if not diff_check.stdout else diff_check.stdout.strip()}")
print("Changed files:")
print(changed_files)